# LTX Video Test: Text-to-Video + Image-to-Video

Prueba para verificar [Lightricks/LTX-Video](https://huggingface.co/Lightricks/LTX-Video).

**Problema conocido:** el modelo completo (2B transformer + T5 encoder + VAE) necesita >12 GB RAM durante la carga porque PyTorch lee cada shard completo a RAM antes de mandarlo a GPU.

**Soluciones en este notebook:**
- Carga con `device_map` (accelerate), que manda los pesos directo a GPU y libera la copia en RAM
- Variante destilada que solo necesita ~4 GB RAM (~6 GB VRAM), 8 steps de inferencia
- FP8 layerwise casting para reducir VRAM del transformer

In [ ]:
# ---- 1. INSTALAR DEPENDENCIAS ----
import os, torch
!pip install -qU diffusers transformers accelerate sentencepiece
!pip install -q imageio[ffmpeg] pillow

In [ ]:
# ---- 2. IMPORTS + DIAGNOSTICO ----
import torch, gc, psutil
from diffusers import LTXPipeline, LTXConditionPipeline, AutoModel
from diffusers.utils import load_image, export_to_video
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from IPython.display import Video

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16

ram_gb = psutil.virtual_memory().total / 1024**3
print(f'RAM del sistema: {ram_gb:.1f} GB')
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    gpu = torch.cuda.get_device_name()
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu} ({vram:.1f} GB VRAM)')
    if ram_gb < 14:
        print('  ADVERTENCIA: <14 GB RAM. El modelo completo podría no cargar.')
        print('  Se recomienda usar la celda de LTX-2.3 Distilled (8 steps).')

## Opción A: Modelo completo Lightricks/LTX-Video

Usa `device_map` para que accelerate cargue cada peso directo a GPU (el pico de RAM se limita al tensor más grande ~unos cientos de MB).

In [ ]:
# ---- 3A. TEXT-TO-VIDEO (modelo completo, device_map) ----

gc.collect()
torch.cuda.empty_cache()

if psutil.virtual_memory().available / 1024**3 < 4:
    print('ERROR: memoria disponible insuficiente. Usa la opcion B (distilled).')
else:
    print('Cargando LTXPipeline con device_map...')
    pipe = LTXPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map={'': DEVICE},
    )
    pipe.vae.enable_tiling()
    print('Pipeline cargado OK')

    prompt = 'A cinematic drone shot of a misty mountain range at sunrise, with golden light.'
    negative_prompt = 'worst quality, inconsistent motion, blurry, jittery, distorted'

    print(f'Generando 33 frames (640x480), 30 steps...')
    with torch.inference_mode():
        output = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            width=640, height=480,
            num_frames=33,
            num_inference_steps=30,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_t2v.mp4', fps=24)
    print('Video guardado: /content/ltx_t2v.mp4')
    Video('/content/ltx_t2v.mp4', embed=True, width=512)

    del pipe
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
# ---- 4A. IMAGE-TO-VIDEO (modelo completo, device_map) ----

gc.collect()
torch.cuda.empty_cache()

if psutil.virtual_memory().available / 1024**3 < 4:
    print('ERROR: memoria insuficiente. Usa opcion B.')
else:
    print('Cargando LTXConditionPipeline con device_map...')
    pipe = LTXConditionPipeline.from_pretrained(
        'Lightricks/LTX-Video',
        torch_dtype=DTYPE,
        device_map={'': DEVICE},
    )
    pipe.vae.enable_tiling()

    image = load_image(
        'https://huggingface.co/datasets/huggingface/documentation-images/'
        'resolve/main/diffusers/guitar-man.png'
    )
    condition = LTXVideoCondition(image=image, frame_index=0)

    print(f'Generando image-to-video...')
    with torch.inference_mode():
        output = pipe(
            prompt='A man playing guitar, slow motion.',
            negative_prompt='worst quality, blurry',
            conditions=[condition],
            width=640, height=480,
            num_frames=33,
            num_inference_steps=30,
            guidance_scale=5.0,
        )

    export_to_video(output.frames[0], '/content/ltx_i2v.mp4', fps=24)
    print('Video guardado: /content/ltx_i2v.mp4')
    Video('/content/ltx_i2v.mp4', embed=True, width=512)

    del pipe; gc.collect(); torch.cuda.empty_cache()

## Opción B: LTX-2.3 Distilled (8 steps, RAM baja)

Modelo destilado de ~2B que funciona con ~4 GB RAM y 6 GB VRAM. 8 steps de inferencia, genera video + audio sincronizado.

In [ ]:
# ---- 3B. LTX 2.3 DISTILLED ----

!pip install -qU git+https://github.com/huggingface/diffusers.git

gc.collect()
torch.cuda.empty_cache()
import psutil
print(f'RAM disponible: {psutil.virtual_memory().available / 1024**3:.1f} GB')

from diffusers import LTX2Pipeline
from diffusers.pipelines.ltx2.export_utils import encode_video

print('Cargando LTX-2.3 Distilled...')
pipe = LTX2Pipeline.from_pretrained(
    'diffusers/LTX-2.3-Distilled-Diffusers',
    torch_dtype=torch.bfloat16,
)
pipe.to('cuda')

prompt = 'A flowing river in a forest at golden hour, gentle wind in the leaves.'
print(f'Generando (8 steps, 768x512, 49 frames)...')

video, audio = pipe(
    prompt=prompt,
    width=768, height=512,
    num_frames=49,
    frame_rate=24,
    num_inference_steps=8,
    guidance_scale=1.0,
    output_type='np',
    return_dict=False,
)

encode_video(
    video[0], fps=24,
    audio=audio[0].float().cpu(),
    audio_sample_rate=pipe.vocoder.config.output_sampling_rate,
    output_path='/content/ltx_fast.mp4',
)
print('Video guardado: /content/ltx_fast.mp4')
Video('/content/ltx_fast.mp4', embed=True, width=512)

del pipe; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 5. LIMPIEZA ----
gc.collect()
torch.cuda.empty_cache()
print('Done')